# Debugger de pipeline SIREN

Ce notebook permet de :
- charger un petit fragment de données (parquet local ou extrait à partir du flux JOCAS si l’environnement est prêt) ;
- construire la pipeline de résolution SIREN ;
- inspecter les requêtes, les résultats et le cache ;
- déboguer facilement une entreprise ou un cas problématique.

In [1]:
from pathlib import Path
import sys
import os 

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

In [2]:
from siren_resolver import (
    Address,
    CompanyQuery,
    CONTRACTING_COLUMNS,
    ParquetSirenCache,
    ResolverConfig,
    SirenResolver,
    GoogleCSEProvider,
    RechercheEntreprisesProvider,
    SirenResolutionPipeline,
)

print('Workspace:', ROOT)
print('Python path ready.')

Workspace: /home/onyxia/work/SirenFinder
Python path ready.


In [3]:
# Configuration de base
DATA_DIR = Path(os.getenv('SIREN_DATA_DIR', ROOT / 'tmp_siren_debug'))
DATA_DIR.mkdir(parents=True, exist_ok=True)

sample_df = None


if sample_df is None:
    try:
        from jocas_common import connect_jocas
        print('Connexion au bucket JOCAS...')
        conn = connect_jocas()
        sample_df = conn.execute(
            "SELECT entreprise_nom, location_label, location_zipcode, location_departement FROM jocas WHERE entreprise_nom IS NOT NULL LIMIT 20"
        ).df()
        sample_df = sample_df.rename(columns={
            'entreprise_nom': 'CONTRACTING_STATED_NAME',
            'location_label': 'CONTRACTING_ADDRESS',
            'location_zipcode': 'location_zipcode',
            'location_departement': 'location_departement',
        })
        print('Fragment JOCAS chargé avec succès.')
    except Exception as exc:
        print(f'Impossible de charger depuis JOCAS : {exc}')

sample_df.head()

Connexion au bucket JOCAS...
Fragment JOCAS chargé avec succès.


,CONTRACTING_STATED_NAME,CONTRACTING_ADDRESS,location_zipcode,location_departement
0,LES FLORALIES,Montauban,82000.0,82
1,EHPAD LA MURELLE,Laurens,34480.0,34
2,DOMALIANCE,Montélimar,26200.0,26
3,HOTEL IGESA,Pralognan la Vanoise,73710.0,73
4,LEO LAGRANGE CENTRE EST,St Didier au Mont d'Or,69370.0,69


In [4]:
# Builder de pipeline et objets de résolution
config = ResolverConfig()
config = ResolverConfig(
    pipeline=config.pipeline,
    recherche_entreprises=config.recherche_entreprises,
    google_cse=config.google_cse,
)

cache = ParquetSirenCache(
    path=DATA_DIR / 'debug_cache.parquet',
    name_col=CONTRACTING_COLUMNS.name_col,
    address_col=CONTRACTING_COLUMNS.address_col,
    siren_col=CONTRACTING_COLUMNS.siren_col,
)

providers = [
    RechercheEntreprisesProvider(config.recherche_entreprises, config.pipeline.min_match_score),
    GoogleCSEProvider(config.google_cse),
]
resolver = SirenResolver(cache=cache, providers=providers)
pipeline = SirenResolutionPipeline(config=config, resolver=resolver, cache=cache, columns=CONTRACTING_COLUMNS)

print('Cache initial:', len(cache))
print('Providers actifs:')
for provider in providers:
    print('-', provider.name, '| available =', provider.is_available)

Aucun cache existant à /home/onyxia/work/SirenFinder/tmp_siren_debug/debug_cache.parquet, démarrage à vide.


Cache initial: 0
Providers actifs:
- recherche_entreprises | available = True
- google_cse | available = False


In [5]:
# Transformer les lignes en CompanyQuery
def to_query(row):
    row_dict = row._asdict()
    raw_name = row_dict[CONTRACTING_COLUMNS.name_col]
    address = CONTRACTING_COLUMNS.build_address(row_dict)
    return CompanyQuery(raw_name=str(raw_name), address=address, role=CONTRACTING_COLUMNS.role)

queries = [to_query(row) for row in sample_df.itertuples(index=False)]
for idx, q in enumerate(queries):
    print(idx, '|', q.raw_name, '|', q.address)

0 | LES FLORALIES | Address(street=None, zip_code='82000', city='Montauban', department_hint='82')
1 | EHPAD LA MURELLE | Address(street=None, zip_code='34480', city='Laurens', department_hint='34')
2 | DOMALIANCE | Address(street=None, zip_code='26200', city='Montélimar', department_hint='26')
3 | HOTEL IGESA | Address(street=None, zip_code='73710', city='Pralognan la Vanoise', department_hint='73')
4 | LEO LAGRANGE CENTRE EST | Address(street=None, zip_code='69370', city="St Didier au Mont d'Or", department_hint='69')
5 | FOYER-LOGEMENT POUR PERSONNES AGEES | Address(street=None, zip_code=None, city='Cantal', department_hint='15')
6 | HOTEL IGESA | Address(street=None, zip_code='73710', city='Pralognan la Vanoise', department_hint='73')
7 | AGENCE L.S.P | Address(street=None, zip_code='69380', city='Alix', department_hint='69')
8 | LA SASSON | Address(street=None, zip_code='73200', city='Albertville', department_hint='73')
9 | MAPAD LA BOISSIERE | Address(street=None, zip_code='69790

In [6]:
# Résolution ligne par ligne pour debug
for q in queries:
    result = resolver.resolve(q)
    print('---')
    print(q.raw_name)
    print('address:', q.address)
    print('siren:', result.siren)
    print('confidence:', result.confidence.value)
    print('match_score:', result.match_score)
    print('matched_name:', result.matched_name)

---
LES FLORALIES
address: Address(street=None, zip_code='82000', city='Montauban', department_hint='82')
siren: 507826675
confidence: official_api
match_score: 1.0
matched_name: LES FLORALIES
---
EHPAD LA MURELLE
address: Address(street=None, zip_code='34480', city='Laurens', department_hint='34')
siren: None
confidence: none
match_score: 0.0
matched_name: None
---
DOMALIANCE
address: Address(street=None, zip_code='26200', city='Montélimar', department_hint='26')
siren: None
confidence: none
match_score: 0.0
matched_name: None
---
HOTEL IGESA
address: Address(street=None, zip_code='73710', city='Pralognan la Vanoise', department_hint='73')
siren: None
confidence: none
match_score: 0.0
matched_name: None
---
LEO LAGRANGE CENTRE EST
address: Address(street=None, zip_code='69370', city="St Didier au Mont d'Or", department_hint='69')
siren: None
confidence: none
match_score: 0.0
matched_name: None
---
FOYER-LOGEMENT POUR PERSONNES AGEES
address: Address(street=None, zip_code=None, city='C

Échec appel API officielle (tentative 1/4) : 400 Client Error: Bad Request for url: https://recherche-entreprises.api.gouv.fr/search?q=EFS+AUVERGNE-LOIRE+SITE+REGIONAL&per_page=5&code_postal=3000&departement=30


---
MAPAD LA BOISSIERE
address: Address(street=None, zip_code='69790', city='St Igny de Vers', department_hint='69')
siren: None
confidence: none
match_score: 0.0
matched_name: None


Échec appel API officielle (tentative 2/4) : 400 Client Error: Bad Request for url: https://recherche-entreprises.api.gouv.fr/search?q=EFS+AUVERGNE-LOIRE+SITE+REGIONAL&per_page=5&code_postal=3000&departement=30
Échec appel API officielle (tentative 3/4) : 400 Client Error: Bad Request for url: https://recherche-entreprises.api.gouv.fr/search?q=EFS+AUVERGNE-LOIRE+SITE+REGIONAL&per_page=5&code_postal=3000&departement=30
Échec appel API officielle (tentative 4/4) : 400 Client Error: Bad Request for url: https://recherche-entreprises.api.gouv.fr/search?q=EFS+AUVERGNE-LOIRE+SITE+REGIONAL&per_page=5&code_postal=3000&departement=30
Fournisseur recherche_entreprises indisponible : recherche-entreprises.api.gouv.fr injoignable : 400 Client Error: Bad Request for url: https://recherche-entreprises.api.gouv.fr/search?q=EFS+AUVERGNE-LOIRE+SITE+REGIONAL&per_page=5&code_postal=3000&departement=30. Passage au suivant.


---
EFS AUVERGNE-LOIRE SITE REGIONAL
address: Address(street=None, zip_code='3000', city='Moulins', department_hint='03')
siren: None
confidence: none
match_score: 0.0
matched_name: None
---
LES OLIVETTES
address: Address(street=None, zip_code='84350', city='Courthézon', department_hint='84')
siren: 508351483
confidence: official_api
match_score: 0.719047619047619
matched_name: LES OLIVETTES (LES OLIVETTES)
---
INFINIGLASS
address: Address(street=None, zip_code='83140', city='Six Fours les Plages', department_hint='83')
siren: 549501187
confidence: official_api
match_score: 1.0
matched_name: INFINIGLASS
---
Chonville Juliette
address: Address(street=None, zip_code='97240', city='Le François', department_hint='972')
siren: None
confidence: none
match_score: 0.0
matched_name: None
---
CARO BEACH VILLAGE
address: Address(street=None, zip_code='97427', city="L'Étang Salé", department_hint='974')
siren: 398449645
confidence: official_api
match_score: 1.0
matched_name: CARO BEACH VILLAGE
---

In [7]:
# Exécution de la pipeline complète sur l'échantillon
sample_path = DATA_DIR / 'sample_contracting_debug.parquet'
sample_df.to_parquet(sample_path, index=False)

resolved_path = DATA_DIR / 'resolved_contracting_debug.parquet'
resolved_df = pipeline.run(missing_input_path=sample_path, resolved_output_path=resolved_path)

print('Résultat sauvegardé dans :', resolved_path)
resolved_df.head()

KeyboardInterrupt: 